# 07b. Random Forest Hyperparameter Tuning - Automobile Loan Default Prediction

Tunes the Random Forest baseline from `05_baseline_models_random_forest.ipynb`. Every decision in this notebook (parameters, class weighting, operating threshold) is made with stratified 5-fold cross-validation on the training split. The held-out test rows are discarded at load time and never used here.


## 1. Setup


In [1]:
import os
import sys

project_root = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.path.abspath(".")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd
from sklearn.metrics import precision_recall_curve
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_val_predict

from src.config import RANDOM_STATE, TARGET
from src.data.data_cleaning import clean_and_split
from src.evaluation.metrics import evaluate_model, log_result_to_csv
from src.models.random_forest import build_random_forest_pipeline

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

train_df, _ = clean_and_split()  # held-out test rows are discarded here
X_train, y_train = train_df.drop(columns=[TARGET]), train_df[TARGET]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results_dir = "experiments/results" if os.path.exists("experiments/results") else "../experiments/results"
experiments_path = os.path.join(results_dir, "experiments.csv")

print("X_train shape:", X_train.shape)
print("Default rate (train):", round(y_train.mean(), 4))


X_train shape: (95372, 40)
Default rate (train): 0.081


## 2. Search Space

- `n_estimators`: more trees reduce variance but cost time.
- `max_depth`: limits how deep each tree grows (`None` means unlimited).
- `min_samples_leaf`: minimum applicants per leaf; larger values smooth the probabilities.
- `max_features`: features considered per split; lower values decorrelate the trees.
- `class_weight`: the imbalance comparison - no weighting vs. `balanced` vs. `balanced_subsample`.


In [2]:
param_distributions = {
    "model__n_estimators": [200, 300, 500],
    "model__max_depth": [None, 8, 12, 16, 20],
    "model__min_samples_leaf": [1, 5, 10, 20, 50],
    "model__max_features": ["sqrt", "log2", 0.3, 0.5],
    "model__class_weight": [None, "balanced", "balanced_subsample"],
}


## 3. Randomized Search

20 sampled configurations, each evaluated with stratified 5-fold CV, scored by ROC-AUC. Each fold refits the full pipeline, including the feature-selection step, so the search cannot leak information from the validation folds. Expect this cell to take 20-30 minutes.


In [3]:
search = RandomizedSearchCV(
    build_random_forest_pipeline(),
    param_distributions=param_distributions,
    n_iter=20,
    scoring="roc_auc",
    cv=skf,
    random_state=RANDOM_STATE,
    n_jobs=1,
    refit=True,
    verbose=1,
)
search.fit(X_train, y_train)

print("Best CV ROC-AUC:", round(search.best_score_, 4))
print("Best parameters:", search.best_params_)


Fitting 5 folds for each of 20 candidates, totalling 100 fits


Best CV ROC-AUC: 0.764
Best parameters: {'model__n_estimators': 500, 'model__min_samples_leaf': 5, 'model__max_features': 0.5, 'model__max_depth': 20, 'model__class_weight': 'balanced'}


## 4. Search Results

The top configurations, and the best score for each class-weighting option. The second table is the imbalance comparison.


In [4]:
results = pd.DataFrame(search.cv_results_)
param_cols = [c for c in results.columns if c.startswith('param_')]
cols = ["rank_test_score", "mean_test_score", "std_test_score"] + param_cols
print(results.sort_values("rank_test_score")[cols].head(10).to_string(index=False))

print()
print("Best CV ROC-AUC per class_weight option:")
print(results.groupby("param_model__class_weight", dropna=False)["mean_test_score"].max())


 rank_test_score  mean_test_score  std_test_score  param_model__n_estimators  param_model__min_samples_leaf param_model__max_features param_model__max_depth param_model__class_weight
               1         0.763993        0.003269                        500                              5                       0.5                     20                  balanced
               2         0.761678        0.003162                        200                              5                       0.5                     20                  balanced
               3         0.760430        0.003379                        500                             10                       0.5                     16                  balanced
               4         0.760230        0.004077                        200                             10                      log2                     16        balanced_subsample
               5         0.757325        0.003384                        500         

## 5. Operating Threshold (out-of-fold, training split only)

At the default 0.5 threshold, the baseline Random Forest catches only 16% of defaulters (recall 0.163). The threshold is chosen from out-of-fold predictions of the tuned model: the highest-precision threshold that still reaches a target recall of 0.65.

Limitation: the threshold is chosen on the same folds whose metrics are reported in section 6, so the reported recall and precision are mildly optimistic. A nested cross-validation would remove that bias.


In [5]:
best_pipeline = search.best_estimator_
oof_proba = cross_val_predict(best_pipeline, X_train, y_train, cv=skf, method="predict_proba", n_jobs=1)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_train, oof_proba)
threshold_table = pd.DataFrame({"threshold": thresholds, "precision": precision[:-1], "recall": recall[:-1]})
threshold_table["f1"] = 2 * threshold_table["precision"] * threshold_table["recall"] / (
    threshold_table["precision"] + threshold_table["recall"] + 1e-12
)

TARGET_RECALL = 0.65
eligible = threshold_table[threshold_table["recall"] >= TARGET_RECALL]
chosen = eligible.sort_values("precision", ascending=False).iloc[0]
OPERATING_THRESHOLD = float(chosen["threshold"])

print(f"Chosen operating threshold: {OPERATING_THRESHOLD:.4f}")
print(f"Out-of-fold recall: {chosen['recall']:.3f}, precision: {chosen['precision']:.3f}")

best_params_path = os.path.join(results_dir, "random_forest_best_parameters.csv")
rows = [{"parameter": k, "value": v} for k, v in search.best_params_.items()]
rows.append({"parameter": "operating_threshold", "value": OPERATING_THRESHOLD})
pd.DataFrame(rows).to_csv(best_params_path, index=False)
print(f"Saved best parameters to {best_params_path}")


Chosen operating threshold: 0.3089
Out-of-fold recall: 0.650, precision: 0.181
Saved best parameters to ../experiments/results\random_forest_best_parameters.csv


## 6. Tuned Random Forest at the Operating Threshold

5-fold CV of the tuned pipeline at the chosen threshold, logged next to the baseline row.


In [6]:
rf_tuned_results = evaluate_model(
    best_pipeline,
    X_train,
    y_train,
    model_name="random_forest_tuned",
    cv=5,
    threshold=OPERATING_THRESHOLD,
)

for key, value in rf_tuned_results.items():
    print(f"{key}: {value}")

log_result_to_csv(rf_tuned_results, experiments_path)
print(f"Logged to {experiments_path}")


model_name: random_forest_tuned
cv_folds: 5
threshold: 0.3088886830691839
roc_auc_mean: 0.7639929485389535
roc_auc_std: 0.0032688045555365735
pr_auc_mean: 0.31330837896117475
pr_auc_std: 0.008629739854076534
recall_mean: 0.6500516457903649
precision_mean: 0.18103522893436702
f1_mean: 0.2831765573542042
confusion_matrix: [[64931, 22717], [2703, 5021]]
fit_seconds: 173.37
Logged to ../experiments/results\experiments.csv


**Finding:** The randomized search (20 configurations, stratified 5-fold CV, ROC-AUC) reached a best score of 0.764, below the untuned baseline's 0.774, so the search did not improve ranking. The sampled configurations did not include the exact baseline setting (300 trees, unlimited depth, min_samples_leaf 1), which may explain the gap. Class weighting: balanced 0.764, balanced_subsample 0.760, no weighting 0.753.

At the operating threshold of 0.309 (chosen for out-of-fold recall of 0.65), the tuned model reaches recall 0.650 and precision 0.181 (F1 0.283). The baseline at 0.5 has recall 0.163, precision 0.632 (F1 0.260). The recall gain costs many more false alarms: 22,717 across the training CV, versus 738 for the baseline.

**Decision:** The best parameters and operating threshold are saved to `experiments/results/random_forest_best_parameters.csv`. `src/models/random_forest.py` is not changed yet. The final Random Forest configuration and threshold stay open until the search is widened to include the baseline setting and the team agrees on the recall target.
